# Benchmark

Loads committed `results/*.json` (produced by `main.py --collect` on a Modal GPU) and renders writeup figures.

In [1]:
import json
from pathlib import Path

import altair as alt
import pandas as pd

RESULTS = Path("results")
CHARTS = Path("artifacts/charts")
CHARTS.mkdir(parents=True, exist_ok=True)

def load(stem):
    hits = sorted(RESULTS.glob(f"{stem}_*.json"))
    return json.loads(hits[-1].read_text()) if hits else None

def save(chart, name):
    chart.save(str(CHARTS / f"{name}.json"))
    return chart

backend_scale = alt.Scale(domain=["contiguous", "paged", "triton", "hf"])
MODELS = ["qwen3-0.6b", "qwen3-4b"]

## E1: Backend × prompt length

In [13]:
rows = []
for m in MODELS:
    d = load(f"e1_{m}")
    if not d:
        continue
    for run in d["runs"]:
        for be, r in run["backends"].items():
            rows.append({
                "model": m, "prompt_len": run["prompt_len"], "backend": be,
                "ttft_ms": 1e3 * r["ttft_s"], "output_tps": r["output_tps"],
                "itl_p50_ms": 1e3 * r["itl"]["p50"] if "itl" in r else None,
                "itl_p90_ms": 1e3 * r["itl"]["p90"] if "itl" in r else None,
                "itl_p99_ms": 1e3 * r["itl"]["p99"] if "itl" in r else None,
            })
e1 = pd.DataFrame(rows)
e1

,model,prompt_len,backend,ttft_ms,output_tps,itl_p50_ms,itl_p90_ms,itl_p99_ms
0,qwen3-0.6b,128,contiguous,48.627699,96.292500,41.365796,41.943425,43.657761
1,qwen3-0.6b,128,paged,103.613939,64.553039,60.936411,63.123198,78.103066
2,qwen3-0.6b,128,triton,67.065180,126.947664,31.154822,31.758653,32.008836
3,qwen3-0.6b,128,hf,182.396521,133.119901,NaN,NaN,NaN
4,qwen3-0.6b,512,contiguous,52.570106,94.145483,42.603285,43.232652,43.471305
5,qwen3-0.6b,512,paged,67.227949,66.233578,60.229938,61.161916,62.386879
6,qwen3-0.6b,512,triton,67.445060,126.340802,31.332582,31.787718,32.708408
7,qwen3-0.6b,512,hf,43.423803,143.800339,NaN,NaN,NaN
8,qwen3-0.6b,1024,contiguous,110.118803,95.293283,41.477631,41.867320,42.175555
9,qwen3-0.6b,1024,paged,85.809289,64.727055,61.470466,63.101539,64.528940


In [3]:
if len(e1):
    prefill = alt.Chart(e1, width=280, height=220).mark_line(point=True).encode(
        x=alt.X("prompt_len:Q", title="prompt tokens"),
        y=alt.Y("ttft_ms:Q", title="time to first token (ms)"),
        color=alt.Color("backend:N", scale=backend_scale),
        column="model:N",
    ).properties(title="Prefill latency")
    display(save(prefill, "e1_prefill_latency"))

alt.Chart(...)

In [4]:
if len(e1):
    decode = alt.Chart(e1, width=280, height=220).mark_line(point=True).encode(
        x=alt.X("prompt_len:Q", title="prompt tokens"),
        y=alt.Y("output_tps:Q", title="output tokens/s"),
        color=alt.Color("backend:N", scale=backend_scale),
        column="model:N",
    ).properties(title="Decode throughput")
    display(save(decode, "e1_decode_throughput"))

alt.Chart(...)

In [5]:
mine1 = e1[e1.backend != "hf"] if len(e1) else e1
if len(mine1):
    band = alt.Chart(mine1, width=280, height=220).mark_area(opacity=0.15).encode(
        x="prompt_len:Q",
        y=alt.Y("itl_p90_ms:Q", title="inter-token latency (ms)"),
        y2="itl_p99_ms:Q",
        color=alt.Color("backend:N", scale=backend_scale),
        column="model:N",
    )
    line = alt.Chart(mine1).mark_line(point=True).encode(
        x="prompt_len:Q", y="itl_p50_ms:Q",
        color=alt.Color("backend:N", scale=backend_scale), column="model:N",
    )
    itl = (band + line).properties(title="Decode latency (p50 line, p90–p99 band)")
    display(save(itl, "e1_itl_bands"))

alt.FacetChart(...)

## E2: Batch-size sweep (weight-traffic amortization)

In [6]:
rows = []
for m in MODELS:
    d = load(f"e2_{m}")
    if not d:
        continue
    for run in d["runs"]:
        for be, r in run["backends"].items():
            rows.append({"model": m, "batch": run["batch_size"], "backend": be,
                         "output_tps": r["output_tps"]})
e2 = pd.DataFrame(rows)
if len(e2):
    chart = alt.Chart(e2, width=280, height=220).mark_line(point=True).encode(
        x=alt.X("batch:Q", scale=alt.Scale(type="log", base=2), title="batch size"),
        y=alt.Y("output_tps:Q", title="output tokens/s"),
        color=alt.Color("backend:N", scale=backend_scale),
        column="model:N",
    ).properties(title="Throughput vs batch size")
    display(save(chart, "e2_batch_sweep"))

alt.Chart(...)

## E3: Attend-only microbench (kernel scaling, no model)

In [7]:
d = load("e3_qwen3-0.6b")
if d:
    rows = []
    for run in d["runs"]:
        for be in ("gather", "triton"):
            rows.append({"S": run["S"], "backend": be, **run[be]})
    e3 = pd.DataFrame(rows)
    lat = alt.Chart(e3).mark_line(point=True).encode(
        x=alt.X("S:Q", scale=alt.Scale(type="log", base=2), title="cached tokens per request (S)"),
        y=alt.Y("ms:Q", scale=alt.Scale(type="log"), title="attend latency (ms)"),
        color=alt.Color("backend:N", scale=backend_scale),
    ).properties(width=360, height=240, title="Attend latency vs S (decode, B=8)")
    memcpy_line = alt.Chart(pd.DataFrame([{"pct": d["memcpy_pct_peak"]}])).mark_rule(
        strokeDash=[6, 4]).encode(y="pct:Q")
    bw = alt.Chart(e3).mark_line(point=True).encode(
        x=alt.X("S:Q", scale=alt.Scale(type="log", base=2), title="cached tokens per request (S)"),
        y=alt.Y("pct_peak:Q", title="% of spec peak bandwidth"),
        color=alt.Color("backend:N", scale=backend_scale),
    )
    uz = (bw + memcpy_line).properties(
            width=360,
            height=240,
            title="Bandwidth utilization (dashed: measured memcpy)")
    display(save(lat, "e3_attend_latency"))
    display(save(uz, "e3_bandwidth"))

alt.Chart(...)

alt.LayerChart(...)

## E4: Arrival stream: continuous vs static (TTFT + lifecycle)

In [8]:
d = load("e4_qwen3-0.6b")
if d:
    rows, spans = [], []
    for cfg, trace in d["configs"].items():
        for rid, r in trace["requests"].items():
            rows.append({"config": cfg, "ttft_s": r["ttft_s"]})
            spans.append({"config": cfg, "request": int(rid), "phase": "waiting",
                          "start": r["arrival_s"], "end": r["first_token_s"]})
            spans.append({"config": cfg, "request": int(rid), "phase": "running",
                          "start": r["first_token_s"], "end": r["completion_s"]})
    e4, e4s = pd.DataFrame(rows), pd.DataFrame(spans)

    ttft = e4.groupby("config").ttft_s.quantile([0.5, 0.99]).unstack().reset_index()
    ttft.columns = ["config", "p50", "p99"]
    ttft = ttft.melt("config", var_name="percentile", value_name="ttft_s")
    bars = alt.Chart(ttft, width=200, height=240).mark_bar().encode(
        x=alt.X("config:N", title=None, axis=alt.Axis(labelAngle=-20)),
        y=alt.Y("ttft_s:Q", title="time to first token (s)"),
        color=alt.Color("config:N", legend=None),
        column=alt.Column("percentile:N", title=None),
    ).properties(title="TTFT under Poisson arrivals")
    display(save(bars, "e4_ttft"))

alt.Chart(...)

In [9]:
if d:
    gantt = alt.Chart(e4s, width=560, height=180).mark_bar().encode(
        x=alt.X("start:Q", title="time (s)"), x2="end:Q",
        y=alt.Y("request:O", title="request (by arrival)", sort="ascending",
                axis=alt.Axis(labels=False, ticks=False)),
        color=alt.Color("phase:N", scale=alt.Scale(domain=["waiting", "running"])),
        row="config:N",
    ).properties(title="Request lifecycle")
    display(save(gantt, "e4_gantt"))
    summary = e4.groupby("config").ttft_s.quantile([0.5, 0.99]).unstack()
    summary.columns = ["ttft_p50_s", "ttft_p99_s"]
    display(summary)

alt.Chart(...)

,ttft_p50_s,ttft_p99_s
config,,
continuous_gather,4.775730,13.030405
continuous_triton,1.613946,3.757514
static_contiguous,10.140140,20.041808


## E5: Preemption under memory pressure

In [10]:
d = load("e5_qwen3-0.6b")
if d:
    e5 = pd.DataFrame(d["runs"])
    display(e5[["fraction", "preemptions", "tokens_identical_to_unpressured", "completion_follows_arrival"]])
    tps = alt.Chart(e5).mark_line(point=True).encode(
        x=alt.X("fraction:Q", title="pool size (fraction of full)", scale=alt.Scale(reverse=True)),
        y=alt.Y("output_tps:Q", title="output tokens/s"),
    ).properties(width=300, height=220, title="Throughput vs pool size")
    ovh = alt.Chart(e5).mark_line(point=True).encode(
        x=alt.X("fraction:Q", title="pool size (fraction of full)", scale=alt.Scale(reverse=True)),
        y=alt.Y("recompute_overhead_pct:Q", title="recomputed tokens (%)"),
    ).properties(width=300, height=220, title="Recompute overhead")
    display(save(tps | ovh, "e5_preemption"))

,fraction,preemptions,tokens_identical_to_unpressured,completion_follows_arrival
0,1.00,0,True,True
1,0.75,0,True,True
2,0.50,0,True,True
3,0.35,2,False,True
4,0.25,5,False,True
5,0.20,5,False,True


alt.HConcatChart(...)

## E6: Block-size sweep

In [11]:
d = load("e6_qwen3-0.6b")
if d:
    e6 = pd.DataFrame(d["runs"])
    display(e6)
    tps = alt.Chart(e6).mark_line(point=True).encode(
        x=alt.X("block_size:Q", scale=alt.Scale(type="log", base=2), title="block size (tokens)"),
        y=alt.Y("output_tps:Q", title="output tokens/s"),
    ).properties(width=300, height=220, title="Throughput vs block size")
    waste = alt.Chart(e6).mark_bar().encode(
        x=alt.X("block_size:O", title="block size (tokens)"),
        y=alt.Y("waste_pct:Q", title="allocated-but-unused (%)"),
    ).properties(width=300, height=220, title="Memory waste vs block size")
    display(save(tps | waste, "e6_block_size"))

,block_size,waste_pct,output_tps
0,8,0.000000,248.815796
1,16,0.000000,249.927288
2,32,0.000000,251.679945
3,64,0.000000,250.228827
4,128,0.000000,248.698183
5,256,33.333333,209.968517


alt.HConcatChart(...)

## E7: Where does a decode tick go? (profile)

In [12]:
d = load("e7_qwen3-0.6b")
if d:
    e7 = pd.DataFrame(d["ops"])
    def bucket(name):
        n = name.lower()
        if "paged_kernel" in n or "triton" in n: return "attend kernel"
        if any(k in n for k in ("index", "pad", "cat", "stack", "copy_", "to_copy")): return "metadata/store"
        if any(k in n for k in ("linear", "mm", "matmul", "gemm")): return "matmuls"
        if any(k in n for k in ("argmax", "softmax", "embedding", "norm", "silu", "mul", "add")): return "model ops"
        return "other"
    e7["bucket"] = e7.name.map(bucket)
    top = e7.nlargest(15, "self_cuda_us")
    chart = alt.Chart(top).mark_bar().encode(
        x=alt.X("self_cuda_us:Q", title="self GPU time (µs, 20 ticks)"),
        y=alt.Y("name:N", sort="-x", title=None),
        color="bucket:N",
    ).properties(width=480, height=320, title="Top ops per decode tick window")
    display(save(chart, "e7_profile"))
    display(e7.groupby("bucket")[["self_cpu_us", "self_cuda_us"]].sum().sort_values("self_cuda_us", ascending=False))

alt.Chart(...)

,self_cpu_us,self_cuda_us
bucket,,
matmuls,103038.055,97344.109
attend kernel,0.000,43796.702
metadata/store,91102.227,33685.624
model ops,124738.892,28962.362
other,62956.411,18777.211
